# Build - Automatic Dockerfile Generation Agent

In this exercise, we build an agent that inspects an existing repository and produces a working `Dockerfile` for it. The agent iterates: it writes a candidate Dockerfile, runs `docker build`, reads the output, and fixes issues until the image builds successfully.

## Environment Variables & Imports

Before using this notebook, please ensure you have obtained an API Key from your LLM backend and update the `.env` file to include it as follows:

```bash
GOOGLE_API_KEY=<copy API key here>
OPENAI_API_KEY=<copy API key here>
ANTRHOPIC_API_KEY=<copy API key here>
LLAMA_CPP_API_KEY=<copy API key here>
```


In [ ]:
import initialize_notebook # noqa

In [ ]:
import json
import os
import pathlib

import jinja2

from hslu.dlm03.common import backend as backend_lib
from hslu.dlm03.common import chat as chat_lib
from hslu.dlm03.common import presets
from hslu.dlm03.common.displays import ipython_display
from hslu.dlm03.common import tools as tools_lib
from hslu.dlm03.util import file as file_lib

## Parameters

In [ ]:
backend_name = "Gemma 4 26B"
backend_config = presets.CONFIGS_BY_NAME[backend_name]
backend = backend_lib.LLM.from_config(backend_config)

In [ ]:
# Path to the repository we want to containerize.
code_folder = pathlib.Path("/Users/vincent/Development/Bank-Account/")

# Desired image tag for `docker build`.
docker_image_name = "auto-dockerfile:dev"

# MCP Server

We expose filesystem tools and a `docker_build` tool that lets the agent iterate on its Dockerfile until it builds.

In [ ]:
import subprocess

from mcp.server import FastMCP

SERVER = FastMCP()


@SERVER.tool()
def list_files(path: str, glob: str = "**/*") -> list[str]:
    """List the files in `path` that match the given glob expression, returning absolute paths.

    Prefer narrow globs (e.g. `*.py`, `requirements*.txt`) to avoid very large listings.
    """
    root = pathlib.Path(path)
    files = [p for p in root.glob(glob) if p.is_file()]
    if not files:
        return "No files found."
    return [str(p) for p in root.glob(glob) if p.is_file() and ".git" not in str(p)]


@SERVER.tool()
def read_file(path: str) -> file_lib.File | str:
    """Reads the file content for the given path."""
    file_path = pathlib.Path(path)
    if not file_path.exists():
        return "File not found."
    if file_path.is_dir():
        return "Given path is a directory."
    if file_path.is_file():
        return file_lib.File.read_from(file_path)
    return "File could not be read."


@SERVER.tool()
def write_file(path: str, content: str) -> str:
    """Writes `content` to `path`, creating parent directories as needed."""
    file_path = pathlib.Path(path)
    try:
        file_path.parent.mkdir(parents=True, exist_ok=True)
        file_path.write_text(content)
    except Exception as e:
        return f"Failed to write file: {e}"
    return f"Successfully wrote {file_path}"


@SERVER.tool()
def docker_build(context_path: str, dockerfile_path: str, tag: str) -> str:
    """Run `docker build` against the given context and Dockerfile.

    Returns the combined stdout / stderr of the docker CLI. A non-zero exit means the build
    failed - use the output to debug.
    """
    try:
        result = subprocess.run(
            ["docker", "build", "-f", dockerfile_path, "-t", tag, context_path],
            capture_output=True,
            text=True,
            timeout=600,
        )
    except FileNotFoundError:
        return "`docker` CLI not found on PATH."
    except subprocess.TimeoutExpired:
        return "docker build timed out after 10 minutes."
    output = (result.stdout or "") + "\n" + (result.stderr or "")
    print(output)
    return f"exit_code={result.returncode}\n{output[-4000:]}"

In [ ]:
import threading

import uvicorn

PORT = 5000
HOST = "localhost"

RUN_ARGS = {
    "app": SERVER.streamable_http_app,
    "port": PORT,
    "host": HOST,
    "log_level": "warning",
}

MCP_THREAD = threading.Thread(target=uvicorn.run, kwargs=RUN_ARGS, daemon=True)
MCP_THREAD.start()

MCP_URL = f"http://{HOST}:{PORT}/mcp"

# Agentic Harness

In [ ]:
import re
def remove_thoughts(text: str) -> str:
    # Remove well-formed <thought>...</thought> blocks (multiline safe)
    cleaned = re.sub(r"<thought>.*?</thought>", "", text, flags=re.DOTALL | re.IGNORECASE)

    # Optionally handle unclosed <thought> tags (strip until end)
    cleaned = re.sub(r"<thought>.*$", "", cleaned, flags=re.DOTALL | re.IGNORECASE)

    return cleaned.strip()

In [ ]:
system_instructions_template_string = \
"""You are an expert DevOps engineer specialized in containerization.

Given a source repository, your goal is to produce an optimized `Dockerfile` at the root of
the repository that correctly builds and runs the project.

Follow this workflow:
1. Use `list_files` and `read_file` to understand the project - look at the entrypoint, the
   language / framework, the dependency manifests (e.g. `requirements.txt`, `pyproject.toml`,
   `package.json`, `go.mod`, `Cargo.toml`), and any existing `.dockerignore` or CI config.
2. Produce a `Dockerfile` that:
   - uses an appropriate official base image (pinned by version),
   - installs build / runtime dependencies in the right order (cache-friendly),
   - uses multi-stage builds if it reduces the final image size,
   - runs as a non-root user,
   - sets a sensible `WORKDIR`, `CMD`/`ENTRYPOINT`, and `EXPOSE`.
3. Write the file with `write_file` and run `docker_build`.
4. If the build fails, read the error, adjust the Dockerfile, and retry (up to a few times).
Stop as soon as the build succeeds.
"""
system_instructions_template = jinja2.Template(system_instructions_template_string, undefined=jinja2.StrictUndefined)

In [ ]:
user_message_template_string = \
"""Please generate a working Dockerfile for the repository at {{ code_folder }} and verify
it by running `docker build` with tag `{{ docker_image_name }}`."""
user_message_template = jinja2.Template(user_message_template_string, undefined=jinja2.StrictUndefined)

In [ ]:
system_instructions = system_instructions_template.render()
user_message = user_message_template.render(
    code_folder=code_folder, docker_image_name=docker_image_name,
)

chat_display = ipython_display.IPythonChatDisplay()
chat_display.show()
chat = chat_lib.Chat(
    messages=[
        {"role": "system", "content": system_instructions},
        {"role": "user", "content": user_message},
    ],
    observers=[chat_display],
)
async with tools_lib.mcp_session(MCP_URL) as session:
    mcp_tools = await session.list_tools()
    tools = [tools_lib.tool_from_mcp(tool) for tool in mcp_tools.tools]
    done = False
    while not done:
        response = backend.generate(chat, tools=tools)
        done = True
        message = response.choices[0].message
        if message.content is not None:
            message.content = remove_thoughts(message.content)
        chat.append(message)
        for tool_call in message.tool_calls or ():
            done = False
            tool_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            tool_call_result = await session.call_tool(tool_name, arguments)
            for content in tool_call_result.content:
                tool_call_result_content = tools_lib.tool_call_result_from_mcp(
                    tool_call.id,
                    content,
                )
                chat.append(tool_call_result_content)